### 0) Comentario de prueba (Primer commit)

In [1]:
print("Comentario para crear un commit en el código")

Comentario para crear un commit en el código


### 00) Comentario de prueba (Tercer commit) 

In [2]:
print("Se creó la rama Miguel-viz" )

Se creó la rama Miguel-viz


### 1) Cargar datos
Carga del archivo CSV, se muestran sus dimensiones y primeras filas.

In [3]:

import pandas as pd
import numpy as np


pd.set_option('display.max_columns', None)

# Ajusta la ruta si es necesario
df = pd.read_csv('BikePrices.CSV')

print('Shape:', df.shape)
df.head(3)



Shape: (1061, 8)


,Brand,Model,Selling_Price,Year,Seller_Type,Owner,KM_Driven,Ex_Showroom_Price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0
1,Bajaj,Bajaj ct 100,18000,2017,Individual,1st owner,35000,32000.0
2,Yo,Yo Style,20000,2011,Individual,1st owner,10000,37675.0


### 2) Vista general y tipos de datos
Se revisan los tipos de datos y columnas por tipo.

In [4]:

print('\nTipos por frecuencia:')
print(df.dtypes.value_counts())
list(df.columns)



Tipos por frecuencia:
object     4
int64      3
float64    1
Name: count, dtype: int64


['Brand',
 'Model',
 'Selling_Price',
 'Year',
 'Seller_Type',
 'Owner',
 'KM_Driven',
 'Ex_Showroom_Price']

### 3) Estandarizar nombres de columnas
Se limpian espacios en blanco antes y después de los nombres

In [5]:

# Quitar únicamente espacios iniciales y finales en los nombres de columnas
df.columns = [c.strip() for c in df.columns]
df.head(1)



,Brand,Model,Selling_Price,Year,Seller_Type,Owner,KM_Driven,Ex_Showroom_Price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0


### 4) Limpieza de texto y datos
Se eliminamn espacios en blanco y normalizan strings.

In [6]:

obj_cols = df.select_dtypes(include=['object']).columns
for c in obj_cols:
    df[c] = df[c].astype(str).str.strip().replace({'': np.nan})
df.head(1)


,Brand,Model,Selling_Price,Year,Seller_Type,Owner,KM_Driven,Ex_Showroom_Price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0


### 5) Conversión de tipos
 Conversión a numérico/fecha cuando es posible.

In [7]:

# Intento masivo a numérico para columnas 'object' (sin forzar)
for c in obj_cols:
    cn = pd.to_numeric(df[c], errors='ignore')
    if not isinstance(cn, pd.Series) or cn.dtype == 'O':
        continue
    df[c] = cn

df.head(1)


/var/folders/d5/0sczjlcn1492zh1b7ly4f0nh0000gn/T/ipykernel_15059/850336458.py:3: FutureWarning: errors='ignore' is deprecated and will raise in a future version. Use to_numeric without passing `errors` and catch exceptions explicitly instead
  cn = pd.to_numeric(df[c], errors='ignore')


,Brand,Model,Selling_Price,Year,Seller_Type,Owner,KM_Driven,Ex_Showroom_Price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0


### 6) Resumen estadístico y cardinalidad
Calcula estadísticos básicos y cantidad de valores únicos por columna.

In [8]:
# Convertir fechas temporalmente a numérico para el describe
df_temp = df.copy()
for col in df_temp.select_dtypes(include=['datetime']):
    df_temp[col] = df_temp[col].astype('int64')  # nanosegundos desde epoch

# Resumen estadístico
display(df_temp.describe(include='all').transpose())

# Cardinalidad
print('\nCardinalidad:')
card = df.nunique(dropna=False).sort_values(ascending=False)
card


,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Brand,1061,20,Bajaj,260,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Model,1061,276,Bajaj Pulsar 150,41,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Selling_Price,1061.0,NaN,NaN,NaN,59638.151744,56304.291973,5000.0,28000.0,45000.0,70000.0,760000.0
Year,1061.0,NaN,NaN,NaN,2013.867107,4.301191,1988.0,2011.0,2015.0,2017.0,2020.0
Seller_Type,1061,2,Individual,1055,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Owner,1061,4,1st owner,924,NaN,NaN,NaN,NaN,NaN,NaN,NaN
KM_Driven,1061.0,NaN,NaN,NaN,34359.833176,51623.152702,350.0,13500.0,25000.0,43000.0,880000.0
Ex_Showroom_Price,626.0,NaN,NaN,NaN,87958.714058,77496.587189,30490.0,54852.0,72752.5,87031.5,1278000.0



Cardinalidad:


KM_Driven            304
Model                276
Ex_Showroom_Price    231
Selling_Price        130
Year                  28
Brand                 20
Owner                  4
Seller_Type            2
dtype: int64

### 7) Valores faltantes
Calcula el conteo y porcentaje de datos faltantes en cada fila

In [9]:
# Conteo y porcentaje de valores faltantes
na_counts = df.isna().sum().sort_values(ascending=False)
na_pct = (df.isna().mean() * 100).sort_values(ascending=False)

# Mostrar top 20 columnas con más faltantes
miss = pd.DataFrame({'na_count': na_counts, 'na_pct': na_pct})
display(miss.head(20))


,na_count,na_pct
Ex_Showroom_Price,435,40.999057
Brand,0,0.000000
Model,0,0.000000
Selling_Price,0,0.000000
Year,0,0.000000
Seller_Type,0,0.000000
Owner,0,0.000000
KM_Driven,0,0.000000


### 8) Eliminación de duplicados
Se detectan y eliminan filas duplicadas exactas. Se reporta el nuevo Shape

In [10]:

dups = df.duplicated().sum()
print('Duplicados encontrados:', dups)
if dups > 0:
    df = df.drop_duplicates().reset_index(drop=True)
print('Shape tras quitar duplicados:', df.shape)


Duplicados encontrados: 6
Shape tras quitar duplicados: (1055, 8)


### 9) Eliminación de Outliers (IQR)
Se seleccionan las columnas numéricas, Calcula Q1 (percentil 25%) y Q3 (percentil 75%) y después Calcula el IQR (rango intercuartílico). Se acotan los valores extremos a [Q1-1.5*IQR, Q3+1.5*IQR].

In [11]:

num_cols = df.select_dtypes(include=['number']).columns
for c in num_cols:
    q1 = df[c].quantile(0.25)
    q3 = df[c].quantile(0.75)
    iqr = q3 - q1
    if pd.isna(iqr) or iqr == 0:
        continue
    low, high = q1 - 1.5*iqr, q3 + 1.5*iqr
    df[c] = df[c].clip(lower=low, upper=high)
df[num_cols].describe().transpose().head(10)


,count,mean,std,min,25%,50%,75%,max
Selling_Price,1055.0,54744.150711,35550.353741,5000.0,28000.0,45000.0,70000.0,133000.00
Year,1055.0,2013.946919,4.022591,2002.0,2011.0,2015.0,2017.0,2020.00
KM_Driven,1055.0,30294.309005,22101.921628,350.0,13500.0,25000.0,43000.0,87250.00
Ex_Showroom_Price,622.0,77782.195740,27776.912903,30490.0,54852.0,72752.5,87031.5,135300.75


### 10) Codificación ligera y guardado
Cambio de categorías de baja cardinalidad a `category` y se guardan en un dataset limpio.

In [13]:

# Categorización ligera (<=50 categorías)
cat_candidates = [c for c in df.columns if df[c].dtype == 'object' and df[c].nunique() <= 50]
for c in cat_candidates:
    df[c] = df[c].astype('category')

# Guardamos
output_path = 'BikePrices_clean.csv'
df.to_csv(output_path, index=False)
print('Archivo guardado en:', output_path)
df.head(3)


Archivo guardado en: BikePrices_clean.csv


,Brand,Model,Selling_Price,Year,Seller_Type,Owner,KM_Driven,Ex_Showroom_Price
0,TVS,TVS XL 100,30000,2017,Individual,1st owner,8000,30490.0
1,Bajaj,Bajaj ct 100,18000,2017,Individual,1st owner,35000,32000.0
2,Yo,Yo Style,20000,2011,Individual,1st owner,10000,37675.0


In [ ]:
# media expresada por marcas
mean_por_marca = df.groupby("Brand").mean(numeric_only=True).round(0)
mean_por_marca

/var/folders/d5/0sczjlcn1492zh1b7ly4f0nh0000gn/T/ipykernel_15059/1640434079.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  mean_por_marca = df.groupby("Brand").mean(numeric_only=True).round(0)


,Selling_Price,Year,KM_Driven,Ex_Showroom_Price
Brand,,,,
Activa,34000.0,2016.0,29683.0,52333.0
Aprilia,70000.0,2018.0,5000.0,NaN
BMW,133000.0,2018.0,2500.0,135301.0
Bajaj,44649.0,2013.0,33458.0,72429.0
Benelli,133000.0,2017.0,2009.0,135301.0
Harley,133000.0,2014.0,9250.0,135301.0
Hero,34537.0,2012.0,38554.0,65885.0
Honda,44569.0,2015.0,30320.0,67646.0
Hyosung,133000.0,2016.0,16500.0,135301.0
